# 03 — Model Training

Train the **Content Tower** (two-tower architecture with shared weights for an item-item recommender) using triplet margin loss.

- Anchor / positive / negative triples are sampled from the cleaned catalog: two items are positive if they share at least one genre, negative if they share none.
- Optimiser: Adam (`lr=1e-3`); loss: `TripletMarginLoss(margin=0.3)`; epochs: 20; batch size: 256.
- Saves the trained checkpoint and the full item-embedding table.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.model import RecommendationModel
from src.trainer import Trainer, TripletDataset

In [ ]:
data_dir = PROJECT_ROOT / 'data'
features = np.load(data_dir / 'features.npy')
genre_matrix = np.load(data_dir / 'genre_matrix.npy')
items = pd.read_parquet(data_dir / 'items.parquet')
print('features    :', features.shape)
print('genre_matrix:', genre_matrix.shape)
print('items       :', items.shape)

In [ ]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

dataset = TripletDataset(
    features=features,
    genre_matrix=genre_matrix,
    num_triples=20000,
)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=True)
print('Triples:', len(dataset))

In [ ]:
model = RecommendationModel(
    input_dim=features.shape[1],
    hidden_dim=256,
    embed_dim=128,
    dropout=0.3,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = Trainer(model=model, optimizer=optimizer, device=device, margin=0.3)

In [ ]:
EPOCHS = 20
losses = trainer.fit(dataloader, epochs=EPOCHS)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(losses) + 1), losses, marker='o', color='#E50914')
ax.set_title('Training Loss per Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Triplet margin loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
item_embeddings = trainer.build_item_embeddings(features, batch_size=512)
print('Embeddings:', item_embeddings.shape)
print('L2 norm (should be ~1):', float(np.linalg.norm(item_embeddings[0])))

trainer.save_checkpoint(data_dir / 'model.pt')
np.save(data_dir / 'item_embeddings.npy', item_embeddings)
print('Saved data/model.pt and data/item_embeddings.npy')